[<img src="../quantumsymmetry_logo.png" alt="QuantumSymmetry" width="450"/>](https://github.com/dariopicozzi/quantumsymmetry)

> **Note:** if you are running this notebook on Google Colab, the next cell will install `quantumsymmetry` and its dependencies (this might take a few minutes):

In [1]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip -q install quantumsymmetry
    !pip -q install pylatexenc

# Sector-Haar sampling

The same chart that runs VQE and time evolution is also an exact **sampler**: it draws Haar-random states *restricted to a sector* -- the active leaves of the pruned tree -- in closed form, one independent sample per call. Random states on a fixed symmetry sector are the raw material for state designs: estimating average fidelities, classical shadows and randomised benchmarking, or simply seeding an optimiser from an unbiased starting point. Here we draw them and check that they form the design the applications rely on.

## Drawing random states on a sector

We reuse the $H_2$ $(1\uparrow, 1\downarrow)$ sector from the first notebook: four qubits, four active leaves. `sample_sector_haar` returns the chart coordinates $(\boldsymbol\theta, \boldsymbol\omega)$ of each sample, which the *complex* `MinimalCircuit` turns into a state vector. Every sample is normalised and supported only on the sector -- no leakage onto the other 12 basis states.

In [2]:
import numpy as np
from quantumsymmetry import MinimalCircuit, sample_sector_haar

mc = MinimalCircuit.from_particle_number(num_spatial_orbitals=2,
                                         num_particles=(1, 1), complex=True)
K = len(mc.support)
rng = np.random.default_rng(0)

theta, omega = sample_sector_haar(mc.num_qubits, mc.support, n_samples=5, rng=rng)
psi = mc.statevector(theta[0], omega[0])

print('sector leaves :', [f'{s:04b}' for s in mc.support])
print('|amp|^2       :', np.round(np.abs(psi[mc.support]) ** 2, 4))
print('norm          :', round(float(np.linalg.norm(psi)), 12))
print('leak off sector:', float(np.max(np.abs(np.delete(psi, mc.support)))))

sector leaves : ['1010', '1001', '0110', '0101']
|amp|^2       : [0.2615 0.1487 0.1095 0.4804]
norm          : 1.0
leak off sector: 0.0


## An exact 1-design

The states are *uniform* on the sector in the strongest sense: their first moment is the maximally mixed state on the $K$ active leaves,

$$\mathbb{E}\,|\psi\rangle\langle\psi| \;=\; \frac{P}{K},$$

where $P$ is the projector onto the sector. (This is what makes the Monte-Carlo average of any observable converge to its sector trace.) We estimate the first moment over many samples and compare with $P/K$.

In [3]:
N = 20000
theta, omega = sample_sector_haar(mc.num_qubits, mc.support, N, rng=rng)

rho = np.zeros((K, K), dtype=complex)
for s in range(N):
    v = mc.statevector(theta[s], omega[s])[mc.support]
    rho += np.outer(v, v.conj())
rho /= N

print('max |E[rho] - I/K| :', f'{np.max(np.abs(rho - np.eye(K) / K)):.2e}')
print('(Monte-Carlo error shrinks as 1 / sqrt(N))')

max |E[rho] - I/K| : 2.62e-03
(Monte-Carlo error shrinks as 1 / sqrt(N))


The empirical first moment matches $P/K$ to the Monte-Carlo error. A $t$-design reproduces the exact sector average of every degree-$t$ polynomial in the state: the 1-design checked above fixes all *linear* averages (expectation values), while the stronger 2-design fixes the *quadratic* ones (variances, average fidelities, classical-shadow estimators) that randomised protocols need. The paper shows the same closed-form draw is in fact an exact 2-design on the sector, which is what underpins its randomised applications.

### The series

1. <a href="tree_01_welcome.ipynb" />Welcome: the binary-tree ansatz</a>
2. <a href="tree_02_ansatz_and_metric.ipynb" />The ansatz, its pruning and its diagonal metric</a>
3. <a href="tree_03_vqe.ipynb" />Natural-gradient VQE for molecules</a>
4. <a href="tree_04_dynamics.ipynb" />Real-time evolution</a>
5. <a href="tree_05_sampling.ipynb" />Sector-Haar sampling</a>
6. <a href="tree_06_dressing.ipynb" />Dressing the ansatz: a symmetry-adapted Schrieffer-Wolff transformation</a>
7. <a href="tree_07_spin.ipynb" />Spin adaptation: exact total spin on the tree</a>

<p style="text-align: left"> <a href="tree_04_dynamics.ipynb" />&lt; Previous: Real-time evolution</a> </p>
<p style="text-align: right"> <a href="tree_06_dressing.ipynb" />Next: Dressing the ansatz: a symmetry-adapted Schrieffer-Wolff transformation &gt;</a> </p>